# DAGs Basics - Exercises

These exercises will help you practice working with Directed Acyclic Graphs (DAGs) for causal inference.

**Note**: Solutions are available in `dags_solutions.ipynb`.

---

## Setup

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from typing import List, Tuple

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)

In [ ]:
# Helper function from the main notebook
def draw_dag(edges: List[Tuple[str, str]], title: str = "DAG", node_size: int = 3000, 
             font_size: int = 12, figsize: Tuple[int, int] = (8, 6)):
    """
    Draw a directed acyclic graph.
    
    Parameters
    ----------
    edges : List[Tuple[str, str]]
        List of (source, target) tuples representing directed edges
    title : str
        Title for the plot
    node_size : int
        Size of nodes in the graph
    font_size : int
        Font size for node labels
    figsize : Tuple[int, int]
        Figure size (width, height)
    """
    G = nx.DiGraph()
    G.add_edges_from(edges)
    
    plt.figure(figsize=figsize)
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    nx.draw(G, pos, 
            with_labels=True,
            node_color='lightblue',
            node_size=node_size,
            font_size=font_size,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            arrowstyle='->',
            edge_color='gray',
            width=2,
            connectionstyle='arc3,rad=0.1')
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return G

---

## Exercise 1: Draw a DAG for Your Domain (Easy)

**Learning Goal**: Practice identifying causal relationships and representing them as a DAG.

**Task**: 
Choose a research question from your field (or use the example below) and draw a DAG showing:
- Treatment variable
- Outcome variable
- At least 3 other variables (confounders, mediators, or colliders)

Then identify:
1. All confounders (variables you must control for)
2. All mediators (variables you should NOT control for if you want total effect)
3. All colliders (variables you should NOT control for)

**Example Question**: Does online advertising increase sales?

<details>
<summary>Hint 1</summary>
Think about what affects both whether a company advertises AND their sales (confounders).
Examples: company size, product quality, marketing budget.
</details>

<details>
<summary>Hint 2</summary>
Think about the mechanism: how does advertising lead to sales? What's in between?
Example: Advertising → Website Traffic → Sales (Website Traffic is a mediator)
</details>

<details>
<summary>Hint 3</summary>
Colliders are common effects. What might be caused by both advertising and sales?
Example: Profit is affected by both advertising costs and sales revenue.
</details>

In [ ]:
# TODO: Define your edges
# Example structure:
# edges = [
#     ('Variable1', 'Variable2'),
#     ('Variable1', 'Variable3'),
#     ...
# ]

edges = [
    # TODO: Add your edges here
]

# Draw the DAG
# G = draw_dag(edges, title="Your DAG Title")

**TODO**: Identify the variables:

1. **Confounders** (must control):
   - 

2. **Mediators** (must NOT control for total effect):
   - 

3. **Colliders** (must NOT control):
   - 

4. **Correct adjustment set** (variables to control for):
   - 

---

## Exercise 2: Simpson's Paradox (Medium)

**Learning Goal**: Understand how confounding can reverse the direction of an association.

**Task**: 
Create a simulation where:
- The **naive** (unadjusted) correlation between Treatment and Outcome is **positive**
- After controlling for a confounder, the association becomes **negative** (or vice versa)

**DAG Structure**: Treatment ← Confounder → Outcome, Treatment → Outcome

**Steps**:
1. Simulate data following the DAG with a confounder
2. Calculate the naive correlation (without controlling)
3. Calculate the adjusted correlation (controlling for confounder)
4. Show that they have opposite signs
5. Visualize the relationship

<details>
<summary>Hint 1</summary>
The confounder needs to have a STRONG positive effect on one variable and a STRONG negative effect on the other (or vice versa). This is what can reverse the sign.
</details>

<details>
<summary>Hint 2</summary>
Structure your simulation like this:
- Z (confounder) ~ N(0, 1)
- X (treatment) = a*Z + noise
- Y (outcome) = b*Z + c*X + noise

To get reversal, try: a > 0, b < 0 (or opposite), and make |b| much larger than |c|.
</details>

<details>
<summary>Hint 3</summary>
Use LinearRegression from sklearn to estimate:
- Naive: Y ~ X
- Adjusted: Y ~ X + Z

Compare the coefficient on X in both models.
</details>

In [ ]:
from sklearn.linear_model import LinearRegression

# TODO: Simulate data
def simulate_simpsons_paradox(n: int = 1000) -> pd.DataFrame:
    """
    Simulate data where naive and adjusted effects have opposite signs.
    """
    np.random.seed(42)
    
    # TODO: Generate confounder Z
    # z = ...
    
    # TODO: Generate treatment X (depends on Z)
    # x = ...
    
    # TODO: Generate outcome Y (depends on both Z and X)
    # y = ...
    
    return pd.DataFrame({
        'confounder': z,
        'treatment': x,
        'outcome': y
    })

# Generate data
# df = simulate_simpsons_paradox()

# TODO: Calculate naive estimate
# X_naive = df[['treatment']].values
# y = df['outcome'].values
# model_naive = LinearRegression()
# ...

# TODO: Calculate adjusted estimate
# X_adjusted = df[['treatment', 'confounder']].values
# model_adjusted = LinearRegression()
# ...

# TODO: Print and compare
# print(f"Naive coefficient: {coef_naive:.3f}")
# print(f"Adjusted coefficient: {coef_adjusted:.3f}")

In [ ]:
# TODO: Visualize the relationship
# Create a scatter plot colored by the confounder to show the reversal

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: naive relationship
# ...

# Right: relationship colored by confounder
# ...

---

## Exercise 3: M-Bias Structure (Medium)

**Learning Goal**: Understand when controlling for a variable can introduce bias even if it's not on a causal path.

**Background**: 
M-bias (or butterfly bias) occurs in this structure:
```
U1 → Z ← U2
 ↓         ↓
 X         Y
```

Where:
- X is the treatment
- Y is the outcome  
- U1 and U2 are unmeasured variables
- Z is measured and affected by both U1 and U2

**The Problem**: 
- There's no confounding path from X to Y (X ⊥ Y unconditionally)
- But if you control for Z (thinking it might help), you OPEN a path: X ← U1 → Z ← U2 → Y
- Z is a **collider** between U1 and U2!

**Task**:
1. Draw the M-bias DAG
2. Simulate data following this structure
3. Show that X and Y are independent without controlling for Z
4. Show that X and Y become associated (biased) when you control for Z

<details>
<summary>Hint 1</summary>
The DAG edges are:
- U1 → X
- U1 → Z
- U2 → Y  
- U2 → Z

Notice: NO direct path from X to Y!
</details>

<details>
<summary>Hint 2</summary>
Simulation structure:
- U1, U2 ~ N(0, 1) (independent)
- X = U1 + noise
- Y = U2 + noise
- Z = U1 + U2 + noise

This makes X and Y independent, but both affect Z.
</details>

<details>
<summary>Hint 3</summary>
To show the bias:
- Naive model: Y ~ X (should show no association)
- Biased model: Y ~ X + Z (will show spurious association)

The second model conditions on the collider Z!
</details>

In [ ]:
# TODO: Draw the M-bias DAG
edges_mbias = [
    # TODO: Add edges for M-bias structure
]

# G_mbias = draw_dag(edges_mbias, title="M-Bias Structure", figsize=(10, 6))

In [ ]:
# TODO: Simulate M-bias data
def simulate_mbias(n: int = 1000) -> pd.DataFrame:
    """
    Simulate M-bias structure where X and Y are independent,
    but become associated if we condition on Z.
    """
    np.random.seed(42)
    
    # TODO: Generate unmeasured variables U1, U2
    # u1 = ...
    # u2 = ...
    
    # TODO: X depends only on U1
    # x = ...
    
    # TODO: Y depends only on U2
    # y = ...
    
    # TODO: Z depends on both U1 and U2 (it's a collider!)
    # z = ...
    
    return pd.DataFrame({
        'u1': u1,
        'u2': u2,
        'x': x,
        'y': y,
        'z': z
    })

# df_mbias = simulate_mbias()

# TODO: Estimate effects
# Model 1: Y ~ X (correct - no confounding)
# ...

# Model 2: Y ~ X + Z (biased - conditioning on collider)
# ...

# TODO: Print results
# print("Effect estimates:")
# print(f"Without controlling for Z: {coef_no_z:.3f} (correct - near 0)")
# print(f"Controlling for Z: {coef_with_z:.3f} (BIASED!)")

**TODO**: Explain in your own words:

1. Why are X and Y independent in this structure?
   - 

2. Why does controlling for Z create bias?
   - 

3. What's the lesson for causal inference?
   - 

---

## Exercise 4: Finding Multiple Paths (Hard)

**Learning Goal**: Practice identifying all paths between variables and determining which need to be blocked.

**Task**:
Create a complex DAG with:
- A treatment variable (X)
- An outcome variable (Y)
- At least **3 different paths** from X to Y (combination of causal and backdoor paths)

Then:
1. Use NetworkX to find all paths programmatically
2. Classify each path as:
   - Causal path (follows arrow directions)
   - Backdoor path (goes against at least one arrow)
3. Identify the **minimal adjustment set** (minimum variables needed to block all backdoor paths)
4. Verify your answer by checking if controlling for these variables blocks confounding

<details>
<summary>Hint 1</summary>
To create multiple paths, you need:
- Direct path: X → Y
- Mediated path: X → M → Y
- Backdoor paths: X ← Z → Y (or longer chains)

Example: Include 2 confounders and 1 mediator.
</details>

<details>
<summary>Hint 2</summary>
Use the `find_all_paths` function from the main notebook:

```python
def find_all_paths(G, source, target, undirected=True):
    if undirected:
        G_undirected = G.to_undirected()
        paths = list(nx.all_simple_paths(G_undirected, source, target))
    else:
        paths = list(nx.all_simple_paths(G, source, target))
    return paths
```

Run with `undirected=True` to get all paths, then `undirected=False` to get only directed paths.
</details>

<details>
<summary>Hint 3</summary>
To identify backdoor paths:
- Find all paths with `undirected=True`
- Find directed paths with `undirected=False`  
- Backdoor paths = All paths - Directed paths

To block a backdoor path, you need to control for at least one non-collider variable on that path.
</details>

In [ ]:
# TODO: Define your complex DAG with multiple paths
edges_complex = [
    # TODO: Add edges to create at least 3 paths from X to Y
    # Include: direct effect, mediator, and confounders
]

# G_complex = draw_dag(edges_complex, title="Complex DAG with Multiple Paths", figsize=(10, 7))

In [ ]:
# TODO: Implement path-finding function
def find_all_paths(G: nx.DiGraph, source: str, target: str, undirected: bool = True) -> List[List[str]]:
    """
    Find all paths between source and target.
    """
    # TODO: Implement this function
    pass

# TODO: Find all paths and directed paths
# all_paths = find_all_paths(G_complex, 'X', 'Y', undirected=True)
# directed_paths = find_all_paths(G_complex, 'X', 'Y', undirected=False)

# TODO: Print paths
# print("All paths from X to Y:")
# for i, path in enumerate(all_paths, 1):
#     print(f"{i}. {' → '.join(path)}")

# print("\nDirected (causal) paths:")
# for i, path in enumerate(directed_paths, 1):
#     print(f"{i}. {' → '.join(path)}")

**TODO**: Classify your paths:

**Causal paths** (follow arrow directions):
1. 
2. 

**Backdoor paths** (confounding - go against arrows):
1. 
2. 

**Minimal adjustment set** (variables to control for):
- 

**Explanation**: Why does this adjustment set block all backdoor paths?
- 

In [ ]:
# TODO: BONUS - Simulate data and verify your adjustment set
# Create a simulation following your DAG structure
# Show that controlling for your adjustment set removes bias

def simulate_complex_dag(n: int = 1000) -> pd.DataFrame:
    """
    Simulate data following your complex DAG.
    """
    np.random.seed(42)
    
    # TODO: Implement simulation based on your DAG
    pass

# df = simulate_complex_dag()

# TODO: Compare naive vs adjusted estimates
# ...

---

## Reflection Questions

After completing these exercises, reflect on:

1. What's harder: drawing the DAG or deciding what to control for? Why?

2. How does the DAG framework compare to the potential outcomes framework from the previous notebook?

3. In your field, what are the biggest challenges to drawing accurate DAGs?

4. Can you think of a case where someone controlled for the wrong variable and got biased results?

---

**Next Steps**: 
- Check your solutions in `dags_solutions.ipynb`
- Try the interactive DAG tool at http://www.dagitty.net/
- Move on to confounding examples in the next notebook